# Clinical / Metadata Branch — Herpes Zoster Multimodal XAI Project

This notebook builds the **clinical-only** branch of the multimodal framework.

- Uses the existing `train.csv` / `validation.csv` / `test.csv` (case-level split, 78/17/17, seed=42) — **no new split is created**.
- Uses **only** metadata columns actually present in these CSVs (no invented features).
- Preprocessing is **fit on train only** and applied to validation/test.
- Produces a clinical baseline classifier **and** a small clinical encoder that outputs a fixed-size embedding per `case_id`, saved for later fusion with 2048-d ResNet50 image embeddings.
- Validation is used for monitoring/early stopping. **The test set is evaluated exactly once**, in the final evaluation section, and never used for tuning or feature selection.

**Sections:** Setup → Data Loading → Clinical Feature Inspection → Preprocessing → Clinical Baseline (train+val) → Clinical Encoder (train+val) → Embedding Generation → Evaluation (val, then final test) → Saving Outputs


## 0. Setup

`torch`, `scikit-learn`, `pandas`, `numpy`, and `joblib` are preinstalled in Colab, so no package
installation is required (and none is run here — an unpinned `pip install --upgrade` can silently
change library versions between runs, which is a reproducibility risk we avoid).

In [ ]:
import json
import random
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Data Loading

Loads the **existing** split files as-is. In Colab, upload `train.csv`, `validation.csv`, `test.csv`
to the working directory (or mount Drive) before running this cell. **These files are not regenerated
or re-split here.**

In [ ]:
TRAIN_PATH = "data/processed/train.csv"
VAL_PATH   = "data/processed/validation.csv"
TEST_PATH  = "data/processed/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape, "| unique case_id:", train_df["case_id"].nunique())
print("Val:  ", val_df.shape,   "| unique case_id:", val_df["case_id"].nunique())
print("Test: ", test_df.shape,  "| unique case_id:", test_df["case_id"].nunique())

# Sanity checks required by project constraints
assert train_df["case_id"].nunique() == len(train_df), "Duplicate case_id in train!"
assert val_df["case_id"].nunique() == len(val_df), "Duplicate case_id in validation!"
assert test_df["case_id"].nunique() == len(test_df), "Duplicate case_id in test!"

train_ids_raw, val_ids_raw, test_ids_raw = set(train_df.case_id), set(val_df.case_id), set(test_df.case_id)
assert not (train_ids_raw & val_ids_raw), "Leakage between train/val!"
assert not (train_ids_raw & test_ids_raw), "Leakage between train/test!"
assert not (val_ids_raw & test_ids_raw), "Leakage between val/test!"

print("\nLabel balance (train):", train_df["label"].value_counts().to_dict())
print("Label balance (val):  ", val_df["label"].value_counts().to_dict())
print("Label balance (test): ", test_df["label"].value_counts().to_dict())


## 2. Clinical Feature Inspection

Identify which columns are usable clinical/metadata features vs. columns that must be excluded
(identifiers, target-derived, or image-related). Missingness is inspected on **training data only**,
so no split-level decision is influenced by validation or test.

In [ ]:
NON_FEATURE_COLS = [
    "case_id", "label", "condition",
    "image_1_path", "image_2_path", "image_3_path", "num_images",
]

all_cols = train_df.columns.tolist()
candidate_cols = [c for c in all_cols if c not in NON_FEATURE_COLS]

print("Candidate clinical columns (all non-identifier / non-target / non-image columns):")
for c in candidate_cols:
    print(" -", c)


In [ ]:
# Inspect missingness using TRAINING data only.
# Validation and test are not used to make feature-selection decisions.

missing_report = train_df[candidate_cols].isna().sum().sort_values(ascending=False)

print("Missing value counts in TRAINING data (out of", len(train_df), "cases):")
print(missing_report)


**Observation:** `other_symptoms_fever` is missing for every case in the training data — it carries
no information at all, so it is dropped. This is a data-driven decision made from training data only,
not an invented feature.

The remaining columns split into two groups:

1. **Categorical columns** (`age_group`, `sex_at_birth`, `fitzpatrick_skin_type`, `condition_duration`) —
   genuine multi-class fields with some missing values.
2. **Binary presence flags** (`textures_*`, `body_parts_*`, `condition_symptoms_*`, `other_symptoms_chills`,
   `other_symptoms_fatigue`) — SCIN encodes these as `YES` when present and blank/NaN when not marked, i.e.
   NaN structurally means "not reported", not a random missing value.

In [ ]:
CATEGORICAL_COLS = [
    "age_group",
    "sex_at_birth",
    "fitzpatrick_skin_type",
    "condition_duration",
]

BINARY_FLAG_COLS = [
    "textures_raised_or_bumpy",
    "textures_flat",
    "textures_rough_or_flaky",
    "textures_fluid_filled",
    "body_parts_head_or_neck",
    "body_parts_arm",
    "body_parts_palm",
    "body_parts_back_of_hand",
    "body_parts_torso_front",
    "body_parts_torso_back",
    "body_parts_buttocks",
    "body_parts_leg",
    "body_parts_foot_top_or_side",
    "body_parts_foot_sole",
    "condition_symptoms_itching",
    "condition_symptoms_burning",
    "condition_symptoms_pain",
    "other_symptoms_chills",
    "other_symptoms_fatigue",
]

DROPPED_COLS = ["other_symptoms_fever"]  # 100% missing in training data

used = set(CATEGORICAL_COLS) | set(BINARY_FLAG_COLS) | set(DROPPED_COLS)
assert used == set(candidate_cols), f"Column mismatch: {set(candidate_cols) - used}"

print(f"Categorical columns ({len(CATEGORICAL_COLS)}):", CATEGORICAL_COLS)
print(f"Binary flag columns ({len(BINARY_FLAG_COLS)}):", BINARY_FLAG_COLS)
print("Dropped (all-missing) columns:", DROPPED_COLS)


## 3. Preprocessing

- Binary flag columns: deterministic mapping `"YES" -> 1`, anything else (NaN) `-> 0`. No fitting needed.
- Categorical columns: missing values filled with the explicit category `"MISSING"`, then
  `OneHotEncoder(handle_unknown="ignore")` **fit on train only**.
- Final numeric feature matrix is scaled with `StandardScaler` **fit on train only**.
- `case_id` and `label` are carried alongside the feature matrix but never used as features.

In [ ]:
def encode_binary_flags(df, cols):
    out = pd.DataFrame(index=df.index)
    for c in cols:
        out[c] = (df[c].astype(str).str.upper() == "YES").astype(int)
    return out

def prep_categorical(df, cols):
    out = df[cols].copy()
    for c in cols:
        out[c] = out[c].astype(str).where(out[c].notna(), "MISSING")
        out[c] = out[c].replace("nan", "MISSING")
    return out

# Binary flags (no fitting required, same deterministic transform everywhere)
train_bin = encode_binary_flags(train_df, BINARY_FLAG_COLS)
val_bin   = encode_binary_flags(val_df, BINARY_FLAG_COLS)
test_bin  = encode_binary_flags(test_df, BINARY_FLAG_COLS)

# Categorical: fill missing, then one-hot fit on TRAIN ONLY
train_cat_raw = prep_categorical(train_df, CATEGORICAL_COLS)
val_cat_raw   = prep_categorical(val_df, CATEGORICAL_COLS)
test_cat_raw  = prep_categorical(test_df, CATEGORICAL_COLS)

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train_cat_raw)

ohe_feature_names = ohe.get_feature_names_out(CATEGORICAL_COLS)

train_cat = pd.DataFrame(ohe.transform(train_cat_raw), columns=ohe_feature_names, index=train_df.index)
val_cat   = pd.DataFrame(ohe.transform(val_cat_raw),   columns=ohe_feature_names, index=val_df.index)
test_cat  = pd.DataFrame(ohe.transform(test_cat_raw),  columns=ohe_feature_names, index=test_df.index)

# Combine categorical + binary into one ordered feature set
feature_names = list(train_cat.columns) + list(train_bin.columns)

X_train_raw = pd.concat([train_cat, train_bin], axis=1)[feature_names].values
X_val_raw   = pd.concat([val_cat, val_bin], axis=1)[feature_names].values
X_test_raw  = pd.concat([test_cat, test_bin], axis=1)[feature_names].values

# Scale: fit on TRAIN ONLY
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

ids_train = train_df["case_id"].values
ids_val   = val_df["case_id"].values
ids_test  = test_df["case_id"].values

print("Feature matrix shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Number of clinical features:", len(feature_names))


## 4. Clinical Baseline

A small, strongly-regularized **Logistic Regression** baseline — appropriate for 112 cases and a feature
count of this size. Trained on the training set only. Evaluated on **validation only** here; test is
evaluated once, later, in the final evaluation section.

In [ ]:
def evaluate(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

def print_metrics(name, m):
    print(f"--- {name} ---")
    for k, v in m.items():
        print(f"{k:20s}: {v:.4f}")
    print()


In [ ]:
baseline_clf = LogisticRegression(
    C=0.5,
    penalty="l2",
    class_weight="balanced",
    max_iter=1000,
    random_state=SEED,
)
baseline_clf.fit(X_train, y_train)

# Validation evaluation only.
# Test is intentionally held back until the final evaluation section (Section 7).
val_prob_bl = baseline_clf.predict_proba(X_val)[:, 1]
val_pred_bl = (val_prob_bl >= 0.5).astype(int)

baseline_val_metrics = evaluate(y_val, val_pred_bl, val_prob_bl)
print_metrics("Baseline (Logistic Regression) — VALIDATION", baseline_val_metrics)


## 5. Clinical Encoder

A small MLP that maps the clinical feature vector to a compact **clinical embedding**
(default 16-d), trained with a classification head on top so the embedding is discriminative.
Kept deliberately small given only 78 training cases: one hidden layer, dropout, weight decay.
Early stopping is monitored on **validation loss only** — test is never touched during training.

In [ ]:
INPUT_DIM = X_train.shape[1]
HIDDEN_DIM = 32
EMBED_DIM = 16

class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, x):
        embedding = self.encoder(x)
        logit = self.classifier(embedding).squeeze(-1)
        return logit, embedding

torch.manual_seed(SEED)
model = ClinicalEncoder(INPUT_DIM).to(DEVICE)
print(model)


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
X_val_t   = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
y_val_t   = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.BCEWithLogitsLoss()

N_EPOCHS = 200
best_val_loss = float("inf")
best_state = None
patience, patience_counter = 20, 0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits, _ = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()

    # Early stopping monitored on VALIDATION only (never test)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d} | train_loss={loss.item():.4f} | val_loss={val_loss:.4f}")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})")
        break

model.load_state_dict(best_state)
model.eval()
print("Loaded best-on-validation encoder weights.")


## 6. Embedding Generation

`get_clinical_embedding(...)` runs the trained encoder on preprocessed clinical features and returns
the embedding **together with `case_id`** (and `label`, for convenience during evaluation/inspection),
so it can later be joined with the 2048-d ResNet50 image embeddings on `case_id`.

In [ ]:
@torch.no_grad()
def get_clinical_embedding(model, X, case_ids):
    """
    Run the trained clinical encoder and return embeddings paired with case_id.

    Args:
        model: trained ClinicalEncoder (in eval mode)
        X: preprocessed clinical feature matrix (numpy array), already scaled/encoded
           with the SAME fitted scaler/encoder used for training.
        case_ids: array-like of case_id values aligned row-for-row with X.

    Returns:
        pandas.DataFrame with columns: case_id, clinical_embed_0, ..., clinical_embed_{D-1}
    """
    model.eval()
    X_t = torch.tensor(np.asarray(X), dtype=torch.float32).to(DEVICE)
    _, embedding = model(X_t)
    embedding = embedding.cpu().numpy()

    cols = [f"clinical_embed_{i}" for i in range(embedding.shape[1])]
    df = pd.DataFrame(embedding, columns=cols)
    df.insert(0, "case_id", np.asarray(case_ids))
    return df

# Actually generate the embeddings for all three splits.
clinical_train_df = get_clinical_embedding(model, X_train, ids_train)
clinical_val_df   = get_clinical_embedding(model, X_val, ids_val)
clinical_test_df  = get_clinical_embedding(model, X_test, ids_test)

# Attach label for convenience (kept separate from the embedding features themselves).
clinical_train_df.insert(1, "label", y_train)
clinical_val_df.insert(1, "label", y_val)
clinical_test_df.insert(1, "label", y_test)

# Alignment check: embedding rows must correspond exactly to the original case_id order.
assert (clinical_train_df["case_id"].values == train_df["case_id"].values).all()
assert (clinical_val_df["case_id"].values == val_df["case_id"].values).all()
assert (clinical_test_df["case_id"].values == test_df["case_id"].values).all()

print(clinical_train_df.shape, clinical_val_df.shape, clinical_test_df.shape)
clinical_train_df.head()


## 7. Evaluation

First the clinical encoder's classification head is evaluated on **validation only**. Then, in one
final block, **both** the baseline and the encoder are evaluated on **test — exactly once**, with no
tuning based on the result.

In [ ]:
@torch.no_grad()
def predict_encoder(model, X):
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    logits, _ = model(X_t)
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= 0.5).astype(int)
    return preds, probs

# Validation evaluation only. Test remains untouched until the final evaluation block below.
val_pred_enc, val_prob_enc = predict_encoder(model, X_val)
encoder_val_metrics = evaluate(y_val, val_pred_enc, val_prob_enc)
print_metrics("Clinical Encoder — VALIDATION", encoder_val_metrics)


### Final Test Evaluation (run once, here only)

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# Test is evaluated only here, after all training/early-stopping/model
# selection decisions have already been made using train + validation.
# No test results are used for tuning.
# ============================================================

# Baseline
test_prob_bl = baseline_clf.predict_proba(X_test)[:, 1]
test_pred_bl = (test_prob_bl >= 0.5).astype(int)
baseline_test_metrics = evaluate(y_test, test_pred_bl, test_prob_bl)

# Clinical encoder
test_pred_enc, test_prob_enc = predict_encoder(model, X_test)
encoder_test_metrics = evaluate(y_test, test_pred_enc, test_prob_enc)

print_metrics("Baseline (Logistic Regression) — TEST", baseline_test_metrics)
print_metrics("Clinical Encoder — TEST", encoder_test_metrics)


In [ ]:
summary = pd.DataFrame({
    "Baseline_Val": baseline_val_metrics,
    "Baseline_Test": baseline_test_metrics,
    "Encoder_Val": encoder_val_metrics,
    "Encoder_Test": encoder_test_metrics,
}).T
summary


## 8. Saving Outputs

Saves everything under a single `outputs/` root, split into consistent subfolders:
- `outputs/data/clinical_train.csv`, `clinical_validation.csv`, `clinical_test.csv` — `case_id` + `label` + clinical embedding, for later fusion with image embeddings.
- `outputs/models/` — fitted preprocessing objects (`OneHotEncoder`, `StandardScaler`), the baseline model, and the encoder weights.
- `outputs/results/` — feature metadata and the evaluation summary, for reproducibility.

In [ ]:
DATA_DIR = "outputs/clinical/embeddings"
MODELS_DIR = "outputs/clinical/models"
RESULTS_DIR = "outputs/clinical/results"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save clinical embeddings
clinical_train_df.to_csv(f"{DATA_DIR}/clinical_train.csv", index=False)
clinical_val_df.to_csv(f"{DATA_DIR}/clinical_validation.csv", index=False)
clinical_test_df.to_csv(f"{DATA_DIR}/clinical_test.csv", index=False)

# Save preprocessing objects
joblib.dump(ohe, f"{MODELS_DIR}/onehot_encoder.joblib")
joblib.dump(scaler, f"{MODELS_DIR}/feature_scaler.joblib")
joblib.dump(baseline_clf, f"{MODELS_DIR}/baseline_logreg.joblib")

# Save clinical encoder weights
torch.save(model.state_dict(), f"{MODELS_DIR}/clinical_encoder_state_dict.pt")

# Save feature/architecture information
feature_info = {
    "seed": SEED,
    "categorical_cols": CATEGORICAL_COLS,
    "binary_flag_cols": BINARY_FLAG_COLS,
    "dropped_cols": DROPPED_COLS,
    "non_feature_cols": NON_FEATURE_COLS,
    "onehot_feature_names": list(ohe_feature_names),
    "final_feature_names": feature_names,
    "n_features": len(feature_names),
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "encoder_architecture": "Linear(input_dim,32) -> ReLU -> Dropout(0.3) -> Linear(32,16) -> ReLU -> Linear(16,1)",
}
with open(f"{RESULTS_DIR}/clinical_feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=2)

# Save evaluation metrics
with open(f"{RESULTS_DIR}/evaluation_summary.json", "w") as f:
    json.dump({
        "baseline_val": baseline_val_metrics,
        "baseline_test": baseline_test_metrics,
        "encoder_val": encoder_val_metrics,
        "encoder_test": encoder_test_metrics,
    }, f, indent=2)

print("Clinical embeddings and artifacts saved successfully.")
print("Embeddings:", DATA_DIR)
print("Models:    ", MODELS_DIR)
print("Results:   ", RESULTS_DIR)


---
**Notes for the next stage (not implemented here):**
- `clinical_train.csv` / `clinical_validation.csv` / `clinical_test.csv` each contain `case_id`, `label`,
  plus `EMBED_DIM`-dimensional clinical embeddings, aligned to the same `case_id`s as the frozen split.
- Fusion with the 2048-d ResNet50 image embeddings should join on `case_id` — this notebook does not
  perform multimodal fusion.
